In [1]:
from pathlib import Path
import chromadb
from openai import OpenAI

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
def chunk_text(text, size=1000, overlap=200) -> list:
    chunks, start = [], 0

    while start < len(text):
        end = min(start+size, len(text))

        if end < len(text):
            bp = text.rfind("\n\n", start, end)

            if bp == -1:
                bp = text.rfind(". ", start, end)
            if bp > start:
                end = bp+1

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start = end-overlap if end < len(text) else end

    return chunks

In [5]:

file_path = Path("./datasets/text_files/harry_potter_knowledge_base.txt")

text = file_path.read_text(encoding="utf-8")
chunks = chunk_text(text)

In [6]:
print(chunks)

["# Harry Potter and the Philosopher's Stone - Comprehensive Knowledge Base\n\n## Chapter 1: The Boy Who Lived\n\nThe story begins on a dull, grey Tuesday with Mr. and Mrs. Dursley of number four, Privet Drive. Mr. Vernon Dursley was a director at a firm called Grunnings, which made drills. He was a big, beefy man with hardly any neck and a large mustache. Mrs. Petunia Dursley was thin and blonde with nearly twice the usual amount of neck, which came in useful as she spent so much of her time craning over garden fences, spying on the neighbors. The Dursleys had a small son called Dudley and in their opinion there was no finer boy anywhere.", "h came in useful as she spent so much of her time craning over garden fences, spying on the neighbors. The Dursleys had a small son called Dudley and in their opinion there was no finer boy anywhere.\n\nThe Dursleys had everything they wanted, but they also had a secret, and their greatest fear was that somebody would discover it. They didn't thin

In [7]:
import httpx

client = OpenAI(http_client=httpx.Client())
embedding_model = "text-embedding-3-small"

def embed_and_store(chunks, db_path, collection_name):
    chroma = chromadb.PersistentClient(path=str(db_path))

    collection = chroma.get_or_create_collection(
        name = collection_name,
        metadata = {
            "description": "Harry Potter knowledge base"
        }
    )

    embeddings = []

    for i in range(0, len(chunks), 100):
        batch = chunks[i:i+100]

        res = client.embeddings.create(model=embedding_model, input=batch)
        embeddings.extend([x.embedding for x in res.data])

    collection.add(
        ids = [f"chunk_{i}" for i in range(len(chunks))],
        documents=chunks,
        embeddings=embeddings,
        metadatas=[{"chunk_index": i} for i in range(len(chunks))],
    )

    return collection


In [8]:
chroma_db_dir = Path("chromadb")
collection = embed_and_store(chunks, chroma_db_dir, "harry_potter_kb")
collection

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Collection(name=harry_potter_kb)

In [9]:
def retrieve(question, top_k=3):
    q_emb = client.embeddings.create(
        model=embedding_model,
        input=question
    ).data[0].embedding

    res = collection.query(
        query_embeddings=[q_emb],
        n_results=top_k,
        include=["documents"],
    )

    return res["documents"][0]

In [10]:
question = "Why did Uncle Vernon take the family to a hut in the middle of the sea?"
docs = retrieve(question)
docs

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


["dozen, then by the hundred. They came through the chimney, through cracks in the door, even delivered by owls. Uncle Vernon tried everything to stop them, but the letters found Harry wherever he was.\n\nFinally, in a desperate attempt to escape the letters, Uncle Vernon drove the family to a hut on a rock in the middle of the sea. On the stroke of midnight on Harry's eleventh birthday, there was a tremendous bang on the door, and it was blasted open by Hagrid, the giant gamekeeper from Hogwarts. Hagrid was furious to discover that the Dursleys had never told Harry about his magical heritage, about his parents' true fate, or about his acceptance to Hogwarts School of Witchcraft and Wizardry.",
 "Little Whinging, Surrey. There was no stamp, and when Harry turned it over, he saw a purple wax seal bearing a coat of arms with a lion, an eagle, a badger, and a snake surrounding a large letter H.\n\nBefore Harry could open it, Uncle Vernon snatched it away. After reading it, his face went f

In [12]:
def answer(question, docs):
    context = "\n\n---\n\n".join(docs)
    prompt = f"""Answer the question using only the context below
Context:
{context}

Question:
{question}

Answer:"""

    res = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[{
            "role": "user",
            "content": prompt
        }],
    )

    return res.choices[0].message.content

In [13]:
question = "Why did Uncle Vernon take the family to a hut in the middle of the sea?"
docs = retrieve(question)
answer_text = answer(question, docs)
answer_text

'He drove them there in a desperate attempt to escape the Hogwarts letters that kept arriving for Harry no matter what they tried.'